In [ ]:
#**burdaki veriyi bizi daha fazla ilerletmiycek yola devam etmeliyim"***

In [ ]:
#peki hangi gelir grubu şarabı tercıh edıyor 
sorgu_detayli="""
select
case
when Income<30000 then 'alt gelir'
when Income BETWEEN 30000 and 70000 then 'orta gelir'
else 'yüksek gelir' end as gelir_grubu,
complain,
count(*) as müşteri_sayisi,
avg(MntWines) as ort_sarap
from final_marketing_data
group by gelir_grubu, complain
order by gelir_grubu, complain desc
"""
sonuc=pd.read_sql(sorgu_detayli,conn)
print(sonuc)
#sorun şarap değil sarap bir sadakat ürünü gibi görünüyor durum farklı.

    gelir_grubu  Complain  müşteri_sayisi   ort_sarap
0     alt gelir         1               4    9.000000
1     alt gelir         0             366   13.833333
2    orta gelir         1              14  175.500000
3    orta gelir         0            1321  259.411809
4  yüksek gelir         1               2  520.500000
5  yüksek gelir         0             504  642.603175


In [ ]:
sorgu="""
select
complain,
count(*) as kişi_sayisi,
avg(MntWines) as sarap,
avg(MntFruits) as meyve,
avg(MntMeatProducts) as et,
avg(MntFishProducts) as balik,
avg(MntSweetProducts) as tatli,
avg(NumWebVisitsMonth) as site_ziyareti
from final_marketing_data
group by complain
"""
sonuc=pd.read_sql(sorgu,conn)
print(sonuc)
#müşteri kaybı %50 sarap reyonunda burda bir sorun olmalı şikayetler ordan geliyor olabilir
#şikayetci kesimin siteyi bu kadar sık ziyaret etmesi de ilginc şarap ceşitlilik kontrolu şikayet iade geri dönüs küntrolu yapıyor olabilirler.
#bir kampanya yapıcak olursak şarap kısmında yapmazmız uygun olur.

   Complain  kişi_sayisi       sarap      meyve          et     balik  \
0         0         2191  306.534916  26.352807  167.553172  37.76586   
1         1           20  176.700000  25.100000  117.700000  26.70000   

       tatli  site_ziyareti  
0  27.139662        5.31675  
1  18.200000        5.85000  


In [ ]:
sorgu="""
select 
Complain,
Education,
count(*) as kişi_sayisi,
avg(Income) as ortalama_gelir,
avg(MntWines+MntFruits+MntMeatProducts+MntFishProducts+MntSweetProducts+MntGoldProds)  as ortalama_harcama
from final_marketing_data
group by complain,education
"""
sonuc=pd.read_sql(sorgu,conn)
print(sonuc)
#harika bir cıkarım hipotezimiz dogru(h1)harcamadaki dşüş sadece gelir azliğiyla acıklanamaz.ozellikle graduation ve phd müşterilr şikayet ettiğinde çüzdanlarını şirkete kapatmişlar.
# bu 20 kişi küçük bir sayı olarak görünsede yüksek eğitimli ve bilinçli kitlenin kaybedilmesi markanın prestijini ve gelecekteki yüksek gelirli müşteri potansiyeline zarar veriyor.

   Complain   Education  kişi_sayisi  ortalama_gelir  ortalama_harcama
0         0    2n Cycle          195    47715.471795        499.610256
1         0       Basic           54    20306.259259         81.796296
2         0  Graduation         1101    52241.296094        625.264305
3         0      Master          363    52942.256198        610.482094
4         0         PhD          478    56132.926778        676.606695
5         1    2n Cycle            3    41766.333333        491.666667
6         1  Graduation           14    46543.142857        380.285714
7         1      Master            2    48430.500000        480.000000
8         1         PhD            1    39684.000000         81.000000


In [ ]:
#bu 20 kişiye yakından bakalım şirkete neden kuskunler ,önce bir gelir dağilimi yapalım
#Hipotez :şikayet eden müşterilerin harcamalarındaki düşüş ,sadece gelirlerinin az olmasından mı kaynaklanıyor(h0),yoksa markaya küstükleri içim mi(h1)
#cıktı yorumu gelir az olabılır ama harzamanın azalma oranı daha fazla *şikayet edenlerin eğitim durumunada bakabılırız eğitim seviyesi arttıkca şikayet bılıncı artar .
sorgu="""
select count(*) as kişi_sayisi,
avg(Income) as ortalama_gelir,
avg(MntWines+MntFruits+MntMeatProducts+MntFishProducts+MntSweetProducts+MntGoldProds)  as ortalama_harcama
from final_marketing_data
group by complain
"""
sonuc=pd.read_sql(sorgu,conn)
print(sonuc)

   kişi_sayisi  ortalama_gelir  ortalama_harcama
0         2191    52016.567777        609.438613
1           20    45672.400000        392.000000


In [ ]:
#count(*) şikayet eden etmeyen kişi sayısı için eklendi,sonuc ilginc sadece 20 kişi şikayet etmiş %1 lik bir kesim bu şirket için iyi şikeyet oranı az
#ama bu kadar az kişi bile kazancta yarı yarıya bir kayıp orneği ,bu kişilerin şikeyet ettikleri seyler ortakmı onları düzeltmek işimize yarar!! benzer şikayeti olan suskunların kaybını engeller.
query_complaint="""
select   complain,
Count(*) as müşteri_sayisi,
AVG(MntWines+MntFruits+MntMeatProducts+MntFishProducts+MntSweetProducts+MntGoldProds) as ortalama_harcanan_tutar
from final_marketing_data
group by Complain
"""
df_complaint_analysis=pd.read_sql_query(query_complaint,conn)
print(df_complaint_analysis)

   Complain  müşteri_sayisi  ortalama_harcanan_tutar
0         0            2191               609.438613
1         1              20               392.000000


In [ ]:
#2.şikayetciler sadıkmı bu analiz önce sikayet edenlerin toplam harcamasına bakalım
#0 son iki yılda şikayet etmemiş ,1 son iki yıldaa şikayet etmişi temsil eder .
query_complaint="""
select   complain,
AVG(MntWines+MntFruits+MntMeatProducts+MntFishProducts+MntSweetProducts+MntGoldProds) as ortalama_harcanan_tutar
from final_marketing_data
group by Complain
"""
df_complaint_analysis=pd.read_sql_query(query_complaint,conn)
print(df_complaint_analysis)

   Complain  ortalama_harcanan_tutar
0         0               609.438613
1         1               392.000000


In [ ]:
#***ŞİKAYET EDEN MÜŞTERİ KİSMİNA BAKTIM*******
#yorumlar cıktılar güzel!!!

In [ ]:
#1.segmentasyon analizi yapıcaz
#verileri power bı da kullanmayı planlıyorum.
import pandas as pd
import sqlite3
#veri tabanına bağlanalım
conn=sqlite3.connect('ifood_analiz.db')
#sql ile gelir gruplayalım;30 bin altı düşük gelir olsun ,30-70 bin orta,70 bin üstü yüksek gelir olsun
##eger istege bağlı select yanına * koysaydık ',' eklememiz gerekiyordu.!!!
query="""
select 
case 
when Income<30000 then 'alt gelir'
when Income BETWEEN 30000 and 70000 then 'orta gelir'
else 'yüksek gelir' end as gelir_grubu
from final_marketing_data
"""
#dataframe cek
df_segmented=pd.read_sql_query(query,conn)
print(df_segmented['gelir_grubu'].value_counts())


gelir_grubu
orta gelir      1335
yüksek gelir     506
alt gelir        370
Name: count, dtype: int64


In [ ]:
#***TEMİZLENEN VERİLER İLE SEGMENTASYON ANALİZLERİ YAPMAYA BAŞLADIM.

In [ ]:
import sqlite3
#temizlediğimiz veriyi veritabanı dosyası oluşturup yerleştirelim
conn=sqlite3.connect('ifood_analiz.db')
#pandas daki tabloyu sql tablosuna dönüştürücez
df.to_sql('final_marketing_data',conn,if_exists='replace',index=False)
#güvenli şekilde bağlantıyı kapatıcaz
conn.close()
print("işlem basarılı veriler ifood_analiz.db  dosyasına mühürlendi.")

işlem basarılı veriler ifood_analiz.db  dosyasına mühürlendi.


In [27]:
#temizlik yapalım daha tutarli bir dağılım için
#1. yas  temizliği ortalama yasam süresi 70 dersem 86 üstü müşterileri cıkarmayı düsündüm
df=df[df['Year_Birth']>1940]
#2.gelir temizliği 200.000 dolardan az kazananları atalım 
df=df[df['Income']<200000]
#3. güncel özete tekrar bakayım
print(df.describe().T)

                      count          mean           std     min      25%  \
ID                   2211.0   5584.673451   3248.177432     0.0   2814.5   
Year_Birth           2211.0   1968.926730     11.688067  1941.0   1959.0   
Income               2211.0  51959.180461  21532.141688  1730.0  35221.0   
Kidhome              2211.0      0.441882      0.536994     0.0      0.0   
Teenhome             2211.0      0.506106      0.544270     0.0      0.0   
Recency              2211.0     48.998191     28.932407     0.0     24.0   
MntWines             2211.0    305.360470    337.381797     0.0     24.0   
MntFruits            2211.0     26.341474     39.749095     0.0      2.0   
MntMeatProducts      2211.0    167.102216    224.279380     0.0     16.0   
MntFishProducts      2211.0     37.665762     54.778567     0.0      3.0   
MntSweetProducts     2211.0     27.058797     41.096258     0.0      1.0   
MntGoldProds         2211.0     43.943012     51.712383     0.0      9.0   
NumDealsPurc

In [ ]:
#sayısal sütunların istatistiksel özeti
df.describe().T
#cıktıyı inceyeleyim income (gelir) deki 24 boşluğu silmiştk burdada 2216 kişi olduğumuz görunuyor .**133 yasında müsteri mi var min doğum tarihi 1863 burda ciddi bir hata var 
#income gelir max 666.666$ mı bu biraz sacma cok yüksek %75lik dilimden fazla bu yuzden trendi anlamamızı zorlastırıcak
#ortalama et (mntmeatproducts) alımına harcanan para 166 iken biri 1725 harcamış  buyuk ihtimalle toplu et alan biri restorant sahibi faln


,count,mean,std,min,25%,50%,75%,max
ID,2216.0,5588.353339,3249.376275,0.0,2814.75,5458.5,8421.75,11191.0
Year_Birth,2216.0,1968.820397,11.985554,1893.0,1959.00,1970.0,1977.00,1996.0
Income,2216.0,52247.251354,25173.076661,1730.0,35303.00,51381.5,68522.00,666666.0
Kidhome,2216.0,0.441787,0.536896,0.0,0.00,0.0,1.00,2.0
Teenhome,2216.0,0.505415,0.544181,0.0,0.00,0.0,1.00,2.0
Recency,2216.0,49.012635,28.948352,0.0,24.00,49.0,74.00,99.0
MntWines,2216.0,305.091606,337.327920,0.0,24.00,174.5,505.00,1493.0
MntFruits,2216.0,26.356047,39.793917,0.0,2.00,8.0,33.00,199.0
MntMeatProducts,2216.0,166.995939,224.283273,0.0,16.00,68.0,232.25,1725.0
MntFishProducts,2216.0,37.637635,54.752082,0.0,3.00,12.0,50.00,259.0


In [ ]:
#income sütunu için boş olan satırları bir atalim 
df=df.dropna(subset=['Income'])
#kontrol edelim hepsinin 0 olması gerek
print(df.isnull().sum())

ID                     0
Year_Birth             0
Education              0
Marital_Status         0
Income                 0
Kidhome                0
Teenhome               0
Dt_Customer            0
Recency                0
MntWines               0
MntFruits              0
MntMeatProducts        0
MntFishProducts        0
MntSweetProducts       0
MntGoldProds           0
NumDealsPurchases      0
NumWebPurchases        0
NumCatalogPurchases    0
NumStorePurchases      0
NumWebVisitsMonth      0
AcceptedCmp3           0
AcceptedCmp4           0
AcceptedCmp5           0
AcceptedCmp1           0
AcceptedCmp2           0
Response               0
Complain               0
Country                0
dtype: int64


In [ ]:
print(df.isnull().sum())
#hangi sütunlarda eksik veri var diye baktık
#cıktıyı inceleyelim;income sütununda 24 boş eksik değer var burda ister bu boş degerleri sileriz istersekte medyan ile doldurma yaparız.

ID                      0
Year_Birth              0
Education               0
Marital_Status          0
 Income                24
Kidhome                 0
Teenhome                0
Dt_Customer             0
Recency                 0
MntWines                0
MntFruits               0
MntMeatProducts         0
MntFishProducts         0
MntSweetProducts        0
MntGoldProds            0
NumDealsPurchases       0
NumWebPurchases         0
NumCatalogPurchases     0
NumStorePurchases       0
NumWebVisitsMonth       0
AcceptedCmp3            0
AcceptedCmp4            0
AcceptedCmp5            0
AcceptedCmp1            0
AcceptedCmp2            0
Response                0
Complain                0
Country                 0
dtype: int64


In [6]:
import pandas as pd
import sqlite3
#csv dosyasını önce pc den bir okutalım
df=pd.read_csv('marketing_data.csv')
#verinin ilk 5 satırına bakalım
print(df.head())


      ID  Year_Birth   Education Marital_Status   Income   Kidhome  Teenhome  \
0   1826        1970  Graduation       Divorced   84835.0        0         0   
1      1        1961  Graduation         Single   57091.0        0         0   
2  10476        1958  Graduation        Married   67267.0        0         1   
3   1386        1967  Graduation       Together   32474.0        1         1   
4   5371        1989  Graduation         Single   21474.0        1         0   

  Dt_Customer  Recency  MntWines  ...  NumStorePurchases  NumWebVisitsMonth  \
0  2014-06-16        0       189  ...                  6                  1   
1  2014-06-15        0       464  ...                  7                  5   
2  2014-05-13        0       134  ...                  5                  2   
3  2014-05-11        0        10  ...                  2                  7   
4  2014-04-08        0         6  ...                  2                  7   

   AcceptedCmp3  AcceptedCmp4  AcceptedCmp5 